# Chapter 12 — Fine-Tuning Generation Models
### Practice Notebook

*Source: Hands-On Large Language Models, Jay Alammar & Maarten Grootendorst (O'Reilly)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter12/Chapter%2012%20-%20Fine-tuning%20Generation%20Models.ipynb)

---

This notebook is your **practice workspace** for Chapter 12. Every code cell is a stub — implement the solution yourself and run it. Refer to the chapter notes at `notes/ch12-fine-tuning-generation-models.md` for theory refreshers.

---

## Table of Contents

- [Part 0: Theory Warm-Up](#part-0-theory-warm-up)
  - [T1: LoRA Parameter Count Calculator](#t1-lora-parameter-count-calculator)
  - [T2: Linear Quantization From Scratch](#t2-linear-quantization-from-scratch)
  - [T3: DPO Loss From Scratch](#t3-dpo-loss-from-scratch)
- [Part 1: Supervised Fine-Tuning with QLoRA](#part-1-supervised-fine-tuning-with-qlora)
  - [1.1 Dataset Preparation and Chat Templates](#11-dataset-preparation-and-chat-templates)
  - [1.2 Model Quantization — BitsAndBytes Config](#12-model-quantization--bitsandbytes-config)
  - [1.3 LoRA Configuration](#13-lora-configuration)
  - [1.4 Training Arguments](#14-training-arguments)
  - [1.5 Train and Save QLoRA Weights](#15-train-and-save-qlora-weights)
  - [1.6 Merge Adapter and Run Inference](#16-merge-adapter-and-run-inference)
- [Part 2: Evaluating the Fine-Tuned Model](#part-2-evaluating-the-fine-tuned-model)
  - [2.1 Perplexity From Scratch](#21-perplexity-from-scratch)
  - [2.2 Qualitative Comparison](#22-qualitative-comparison)
- [Part 3: Preference Tuning with DPO](#part-3-preference-tuning-with-dpo)
  - [3.1 DPO Dataset Preparation](#31-dpo-dataset-preparation)
  - [3.2 Load Quantized SFT Model](#32-load-quantized-sft-model)
  - [3.3 LoRA Configuration for DPO](#33-lora-configuration-for-dpo)
  - [3.4 DPO Training Configuration](#34-dpo-training-configuration)
  - [3.5 Train with DPOTrainer](#35-train-with-dpotrainer)
  - [3.6 Stack and Merge Both Adapters](#36-stack-and-merge-both-adapters)
  - [3.7 Final Aligned Model Inference](#37-final-aligned-model-inference)

### [OPTIONAL] — Install packages on Colab

Uncomment and run if you are on a cloud GPU environment.

💡 **GPU required.** Go to Runtime → Change runtime type → GPU (T4 on Colab).

In [ ]:
# %%capture
# !pip install -q accelerate==0.31.0 peft==0.11.1 bitsandbytes==0.43.1 \
#              transformers==4.41.2 trl==0.9.4 sentencepiece==0.2.0 datasets

---
# Part 0: Theory Warm-Up

These exercises test the core mathematical ideas from the notes **before** touching any training code. They use only `numpy` and `math` — no GPU needed.

Work through them in order. Each one builds intuition you will need when reading the QLoRA and DPO training code in Parts 1–3.

## T1: LoRA Parameter Count Calculator

LoRA replaces a direct weight update $\Delta W \in \mathbb{R}^{d \times d}$ with the product of two thin matrices $A \in \mathbb{R}^{d \times r}$ and $B \in \mathbb{R}^{r \times d}$.

The parameter saving is:
$$\text{LoRA params} = d \times r + r \times d = 2dr$$
$$\text{Full FT params} = d \times d = d^2$$
$$\text{Compression ratio} = \frac{d^2}{2dr} = \frac{d}{2r}$$

**Task:** Implement `lora_stats(d, r)` that prints:
- Full fine-tuning parameter count for a single `d×d` matrix
- LoRA parameter count for rank `r`
- Compression ratio

Then call it for the following real-world cases:

| Model | d | r |
|-------|---|---|
| TinyLlama (1.1B) | 2,048 | 64 |
| LLaMA-2 7B | 4,096 | 16 |
| GPT-3 175B | 12,288 | 8 |

**Expected output for GPT-3:**
```
d=12288, r=8
  Full FT params:   150,994,944
  LoRA params:          196,608
  Compression:          768.0×
```

In [ ]:
def lora_stats(d: int, r: int) -> None:
    """Print LoRA vs full fine-tuning parameter counts for a d×d weight matrix."""
    # YOUR CODE HERE
    pass


# Call for each model
lora_stats(d=2_048,  r=64)   # TinyLlama
print()
lora_stats(d=4_096,  r=16)   # LLaMA-2 7B
print()
lora_stats(d=12_288, r=8)    # GPT-3 175B

### T1b: Manual LoRA Forward Pass

Implement the actual LoRA forward pass on a tiny example.

Given:
- Frozen weight matrix $W$ (shape 4×4)
- Trainable $A$ (shape 4×1) and $B$ (shape 1×4)
- Input vector $x$ (shape 4)
- Scaling: $\alpha=2, r=1$ → scale factor $= \alpha/r = 2.0$

Compute:
1. The standard output: $Wx$
2. The LoRA update: $\frac{\alpha}{r} \cdot ABx$
3. The combined output: $Wx + \frac{\alpha}{r} \cdot ABx$

**Expected output:**
```
W·x        = [4.  3.  4.  3.]
LoRA (A·B·x) scaled = [0.3 0.18 0.42 0.06]
Combined    = [4.3  3.18  4.42  3.06]
```

In [ ]:
import numpy as np

# Frozen weight matrix W (4×4)
W = np.array([
    [2, 0, 1, 0],
    [0, 3, 0, 1],
    [1, 0, 2, 0],
    [0, 1, 0, 3]
], dtype=float)

# Trainable LoRA matrices
A = np.array([[0.5], [0.3], [0.7], [0.1]])   # shape 4×1
B = np.array([[0.2, 0.4, 0.1, 0.3]])          # shape 1×4

# Input vector
x = np.array([1.0, 1.0, 1.0, 1.0])

alpha = 2
r = 1
scale = alpha / r   # = 2.0

# YOUR CODE HERE
# 1. standard_out = ...
# 2. lora_update  = ...
# 3. combined     = ...

# print(f"W·x             = {standard_out}")
# print(f"LoRA (A·B·x) scaled = {lora_update}")
# print(f"Combined        = {combined}")

---
## T2: Linear Quantization From Scratch

Quantization maps floating-point weights to a grid of integer codes using:

$$q = \text{round}\!\left(\frac{x - x_{\min}}{x_{\max} - x_{\min}} \times (2^b - 1)\right)$$

Dequantization (reconstruction) reverses this:

$$\hat{x} = \frac{q}{2^b - 1} \times (x_{\max} - x_{\min}) + x_{\min}$$

**Task:** Implement `quantize(weights, bits)` and `dequantize(codes, x_min, x_max, bits)`. Then:
1. Quantize the weights `[-1.2, 0.3, 0.7, 2.1]` to 2 bits
2. Dequantize back and compute the reconstruction error for each weight
3. Add an outlier weight `50.0` and re-quantize. Observe how the error explodes for the common weights.

**Expected output (step 1–2):**
```
Weights:     [-1.2,  0.3,  0.7,  2.1]
Codes (2-bit): [0, 1, 2, 3]
Reconstructed: [-1.2, -0.1,  1.0,  2.1]
Errors:        [ 0.0,  0.4,  0.3,  0.0]
```

In [ ]:
import numpy as np


def quantize(weights: np.ndarray, bits: int):
    """
    Quantize float weights to integer codes using linear quantization.
    Returns: (codes, x_min, x_max)
    """
    # YOUR CODE HERE
    # Hint: levels = 2**bits - 1
    pass


def dequantize(codes: np.ndarray, x_min: float, x_max: float, bits: int) -> np.ndarray:
    """
    Reconstruct float weights from integer codes.
    """
    # YOUR CODE HERE
    pass


# --- Step 1-2: normal weights ---
weights = np.array([-1.2, 0.3, 0.7, 2.1])

# YOUR CODE HERE
# codes, x_min, x_max = quantize(weights, bits=2)
# reconstructed = dequantize(codes, x_min, x_max, bits=2)
# errors = np.abs(weights - reconstructed)
# print each result


# --- Step 3: add outlier, re-quantize ---
weights_with_outlier = np.array([-1.2, 0.3, 0.7, 2.1, 50.0])

# YOUR CODE HERE
# Observe: the outlier forces x_max to 50.0, which destroys precision
# for all the common weights near zero

---
## T3: DPO Loss From Scratch

The DPO loss is:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)$$

In practice, $\pi_\theta(y \mid x)$ is the sum of log-probabilities of all tokens in the response.

**Task:** Implement `dpo_loss(log_p_train_chosen, log_p_ref_chosen, log_p_train_rejected, log_p_ref_rejected, beta)` that returns the scalar DPO loss.

Then compute the loss for two scenarios:

| Scenario | Description | Expected loss |
|----------|------------|---------------|
| **Good** | Model already prefers chosen over rejected (relative to ref) | small (~0.47) |
| **Bad** | Model still prefers rejected over chosen (relative to ref) | large (~0.94) |

**Scenario values:**
```
Good:  log_p_train_chosen=-10.0, log_p_ref_chosen=-12.4
       log_p_train_rejected=-12.8, log_p_ref_rejected=-10.1
Bad:   log_p_train_chosen=-13.0, log_p_ref_chosen=-12.4
       log_p_train_rejected=-10.5, log_p_ref_rejected=-10.1
beta=0.1
```

In [ ]:
import math


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


def dpo_loss(
    log_p_train_chosen: float,
    log_p_ref_chosen: float,
    log_p_train_rejected: float,
    log_p_ref_rejected: float,
    beta: float = 0.1,
) -> float:
    """
    Compute the scalar DPO loss for a single (prompt, chosen, rejected) triplet.

    Args:
        log_p_train_chosen:   log P_theta(chosen | prompt)
        log_p_ref_chosen:     log P_ref(chosen | prompt)
        log_p_train_rejected: log P_theta(rejected | prompt)
        log_p_ref_rejected:   log P_ref(rejected | prompt)
        beta: temperature controlling distance from reference model

    Returns:
        Scalar DPO loss value
    """
    # Step 1: compute log-probability ratios (log-space division = subtraction)
    # ratio_chosen   = log_p_train_chosen   - log_p_ref_chosen
    # ratio_rejected = log_p_train_rejected - log_p_ref_rejected

    # Step 2: compute beta * (ratio_chosen - ratio_rejected)

    # Step 3: pass through sigmoid, then take -log

    # YOUR CODE HERE
    pass


# Scenario 1: model already prefers chosen (relative to reference)
loss_good = dpo_loss(
    log_p_train_chosen=-10.0,
    log_p_ref_chosen=-12.4,
    log_p_train_rejected=-12.8,
    log_p_ref_rejected=-10.1,
    beta=0.1,
)
print(f"Loss (model prefers chosen):   {loss_good:.4f}  (expected ~0.47)")

# Scenario 2: model still prefers rejected (relative to reference)
loss_bad = dpo_loss(
    log_p_train_chosen=-13.0,
    log_p_ref_chosen=-12.4,
    log_p_train_rejected=-10.5,
    log_p_ref_rejected=-10.1,
    beta=0.1,
)
print(f"Loss (model prefers rejected): {loss_bad:.4f}  (expected ~0.94)")
print(f"Ratio (bad/good loss):         {loss_bad/loss_good:.2f}×  ← bad loss should be much higher")

---
# Part 1: Supervised Fine-Tuning with QLoRA

We fine-tune **TinyLlama-1.1B** on 3,000 conversations from the UltraChat dataset using QLoRA: the base model is loaded in 4-bit NF4 quantization (frozen), and only the LoRA adapter matrices $A$ and $B$ are trained.

The goal: turn the base model (which only completes text patterns) into an instruction-following model.

## 1.1 Dataset Preparation and Chat Templates

TinyLlama expects a specific chat format to distinguish user turns from assistant turns:
```
<|user|>
[user message]</s>
<|assistant|>
[assistant response]</s>
```

**Task:**
1. Load the `TinyLlama/TinyLlama-1.1B-Chat-v1.0` tokenizer (it carries the chat template)
2. Write `format_prompt(example)` that applies `apply_chat_template` to `example["messages"]` and returns `{"text": prompt}`
3. Load the `HuggingFaceH4/ultrachat_200k` dataset, `split="train_sft"`, shuffle with `seed=42`, select the first 3,000 rows
4. Apply `format_prompt` with `.map()`
5. Print `dataset["text"][0]` to verify the template is applied correctly

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

# 1. Load tokenizer with chat template
# YOUR CODE HERE
# template_tokenizer = AutoTokenizer.from_pretrained(...)

# 2. Format function
def format_prompt(example):
    """Apply TinyLlama chat template to a conversation example."""
    # YOUR CODE HERE
    # Hint: template_tokenizer.apply_chat_template(example["messages"], tokenize=False)
    pass

# 3. Load 3,000 examples
# YOUR CODE HERE
# dataset = (
#     load_dataset(..., split=...)
#     .shuffle(seed=42)
#     .select(range(3_000))
# )

# 4. Apply format
# YOUR CODE HERE
# dataset = dataset.map(format_prompt)

# 5. Inspect
# print(dataset["text"][0])

## 1.2 Model Quantization — BitsAndBytes Config

Load `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T` in 4-bit NF4 quantization.

**Task:** Create a `BitsAndBytesConfig` and load the model. Each parameter has a specific role — make sure you understand what you are setting:

| Parameter | Value | Why |
|-----------|-------|-----|
| `load_in_4bit` | `True` | Store weights as 4-bit integers |
| `bnb_4bit_quant_type` | `"nf4"` | Use NormalFloat bins (distribution-aware) |
| `bnb_4bit_compute_dtype` | `"float16"` | Dequantize to float16 for computation |
| `bnb_4bit_use_double_quant` | `True` | Also quantize the quantization constants |

Also load the tokenizer for the same model (not the chat tokenizer). Set `pad_token = "<PAD>"` and `padding_side = "left"`.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-bit quantization config — the Q in QLoRA
# YOUR CODE HERE
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=...,
#     bnb_4bit_quant_type=...,
#     bnb_4bit_compute_dtype=...,
#     bnb_4bit_use_double_quant=...,
# )

# Load model
# YOUR CODE HERE
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",
#     quantization_config=bnb_config,
# )
# model.config.use_cache = False
# model.config.pretraining_tp = 1

# Load tokenizer
# YOUR CODE HERE
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
# tokenizer.pad_token = ...
# tokenizer.padding_side = ...

## 1.3 LoRA Configuration

**Task:** Create a `LoraConfig` with the following parameters, then call `prepare_model_for_kbit_training` and `get_peft_model`.

| Parameter | Value | What it controls |
|-----------|-------|------------------|
| `lora_alpha` | 32 | Scale factor α (update magnitude = α/r × ΔW) |
| `lora_dropout` | 0.1 | Regularisation on adapter activations |
| `r` | 64 | Rank — size of A and B matrices |
| `bias` | `"none"` | Don't train bias terms |
| `task_type` | `"CAUSAL_LM"` | Causal language modelling |
| `target_modules` | all 7 projections | See list below |

Target modules: `['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']`

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# YOUR CODE HERE
# peft_config = LoraConfig(
#     lora_alpha=...,
#     lora_dropout=...,
#     r=...,
#     bias=...,
#     task_type=...,
#     target_modules=[...]
# )

# Prepare model for k-bit training and attach LoRA adapters
# YOUR CODE HERE
# model = prepare_model_for_kbit_training(model)
# model = get_peft_model(model, peft_config)

## 1.4 Training Arguments

**Task:** Create `TrainingArguments` with the values below. After writing the code, answer the questions underneath.

| Parameter | Value |
|-----------|-------|
| `output_dir` | `"./results"` |
| `per_device_train_batch_size` | 2 |
| `gradient_accumulation_steps` | 4 |
| `optim` | `"paged_adamw_32bit"` |
| `learning_rate` | 2e-4 |
| `lr_scheduler_type` | `"cosine"` |
| `num_train_epochs` | 1 |
| `logging_steps` | 10 |
| `fp16` | `True` |
| `gradient_checkpointing` | `True` |

In [ ]:
from transformers import TrainingArguments

# YOUR CODE HERE
# training_arguments = TrainingArguments(
#     ...
# )

**Comprehension questions — answer in comments:**

1. With `per_device_train_batch_size=2` and `gradient_accumulation_steps=4`, what is the **effective batch size**? Why does this help with memory?

2. What does `gradient_checkpointing=True` trade off? (Hint: think about what is stored during the forward pass.)

3. Why is the learning rate 2e-4 here but will be 1e-5 in Part 3 (DPO)?

In [ ]:
# Your answers (as comments):

# Q1: Effective batch size = ...
#     Why it helps with memory: ...

# Q2: gradient_checkpointing trades ...
#     for ...

# Q3: DPO uses a lower LR because ...

## 1.5 Train and Save QLoRA Weights

**Task:** Create an `SFTTrainer` and call `.train()`. Then save only the LoRA adapter weights.

Key parameters for `SFTTrainer`:
- `model` — the PEFT-wrapped model from 1.3
- `train_dataset` — the formatted UltraChat dataset from 1.1
- `dataset_text_field` — `"text"`
- `tokenizer` — from 1.2
- `args` — from 1.4
- `max_seq_length` — 512
- `peft_config` — from 1.3

**Note:** Training will take ~20–60 minutes on a T4 GPU. Monitor the loss — it should decrease from ~2.0 to ~1.2 over the first epoch.

In [ ]:
from trl import SFTTrainer

# YOUR CODE HERE
# trainer = SFTTrainer(
#     model=...,
#     train_dataset=...,
#     dataset_text_field=...,
#     tokenizer=...,
#     args=...,
#     max_seq_length=512,
#     peft_config=...,
# )

# trainer.train()

# Save only the LoRA adapter (not the full model — much smaller!)
# trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

## 1.6 Merge Adapter and Run Inference

`merge_and_unload()` permanently fuses the LoRA matrices into $W$: it computes $W' = W + \frac{\alpha}{r}AB$ for every targeted layer, then discards the adapter objects. After merging, the model has zero PEFT overhead.

**Task:**
1. Load the saved adapter with `AutoPeftModelForCausalLM.from_pretrained`
2. Call `merge_and_unload()`
3. Run inference with the prompt below
4. Compare the output to what a base model would produce (just repeating the question pattern vs giving an actual answer)

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import pipeline

# Load adapter and merge into base model
# YOUR CODE HERE
# model = AutoPeftModelForCausalLM.from_pretrained(
#     "TinyLlama-1.1B-qlora",
#     low_cpu_mem_usage=True,
#     device_map="auto",
# )
# merged_model = model.merge_and_unload()

# Inference using the TinyLlama chat template
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# YOUR CODE HERE
# pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
# print(pipe(prompt, max_new_tokens=200)[0]["generated_text"])

⚠️ **VRAM Clean-up** — Run the cell below before Part 2.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

---
# Part 2: Evaluating the Fine-Tuned Model

Before moving to preference tuning, let's evaluate what SFT actually changed. This section uses only `math` and `numpy` — no GPU needed for these exercises.

## 2.1 Perplexity From Scratch

Perplexity is defined as:

$$\text{PPL}(W) = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid w_1, \ldots, w_{i-1})\right)$$

**Task:** Implement `perplexity(log_probs)` that takes a list of per-token log-probabilities and returns the perplexity score.

Then compute and compare perplexity for two hypothetical model outputs on the same 4-token sequence:

| Token | Base model log-prob | SFT model log-prob |
|-------|---------------------|--------------------|
| "Large" | -2.3 | -1.2 |
| "language" | -1.8 | -0.9 |
| "models" | -2.1 | -0.8 |
| "are" | -1.5 | -0.6 |

**Expected:**
```
Base model PPL:  ~8.3
SFT model PPL:   ~2.7
```
*Lower perplexity = more confident, more fluent output.*

In [ ]:
import math


def perplexity(log_probs: list) -> float:
    """
    Compute perplexity from a list of per-token log-probabilities.

    Args:
        log_probs: list of log P(w_i | context) for each token

    Returns:
        Perplexity score (lower is better)
    """
    # YOUR CODE HERE
    # Hint: average the log probs, negate, exponentiate
    pass


base_log_probs = [-2.3, -1.8, -2.1, -1.5]
sft_log_probs  = [-1.2, -0.9, -0.8, -0.6]

# YOUR CODE HERE
# print(f"Base model PPL: {perplexity(base_log_probs):.2f}")
# print(f"SFT model PPL:  {perplexity(sft_log_probs):.2f}")

## 2.2 Qualitative Comparison

The best evaluation is to run your own prompts and observe the model's behaviour.

**Task:** Design 3 test prompts that reveal different aspects of the SFT training:
1. A factual question (tests whether the model actually answers vs. continues a pattern)
2. A formatting request (e.g., "list 3 examples of..." — tests instruction-following)
3. A question that requires reasoning (e.g., "why does X happen?")

Write the prompts below and describe what you'd expect a base model to produce vs. an instruction-tuned model.

In [ ]:
# Define your 3 test prompts using the TinyLlama chat template
TEMPLATE = """<|user|>
{question}</s>
<|assistant|>
"""

prompt_1 = TEMPLATE.format(question="YOUR FACTUAL QUESTION HERE")
prompt_2 = TEMPLATE.format(question="YOUR FORMATTING INSTRUCTION HERE")
prompt_3 = TEMPLATE.format(question="YOUR REASONING QUESTION HERE")

# If your merged model is still in memory, run:
# for p in [prompt_1, prompt_2, prompt_3]:
#     print(pipe(p, max_new_tokens=150)[0]["generated_text"])
#     print("-" * 60)

---
# Part 3: Preference Tuning with DPO

SFT taught the model to follow instructions. DPO will now teach it to prefer *better* responses over *worse* ones — without a separate reward model.

The training data is `{prompt, chosen, rejected}` triplets. The loss pushes the model to simultaneously:
- Increase its relative preference for the chosen response (vs. the reference model's baseline)
- Decrease its relative preference for the rejected response

## 3.1 DPO Dataset Preparation

We use `argilla/distilabel-intel-orca-dpo-pairs` — a preference dataset where each row has a prompt, a chosen response (labelled preferred), and a rejected response.

**Task:**
1. Write `format_prompt(example)` for the DPO format — it must return a dict with three keys: `"prompt"`, `"chosen"`, `"rejected"`. The prompt should include a system message prefix.
2. Load the dataset
3. Filter out: ties (`status == "tie"`), zero-scored choices (`chosen_score == 0`), and GSM8k contamination (`in_gsm8k_train == True`)
4. Apply `format_prompt` and remove all original columns
5. Print the number of remaining rows and inspect one example

In [ ]:
from datasets import load_dataset


def format_prompt(example):
    """Format a DPO example for TinyLlama's chat template."""
    # YOUR CODE HERE
    # system = "<|system|>\n" + example['system'] + "</s>\n"
    # prompt = "<|user|>\n" + example['input'] + "</s>\n<|assistant|>\n"
    # chosen   = example['chosen']   + "</s>\n"
    # rejected = example['rejected'] + "</s>\n"
    # return {"prompt": ..., "chosen": ..., "rejected": ...}
    pass


# Load dataset
# YOUR CODE HERE
# dpo_dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")

# Filter
# YOUR CODE HERE
# dpo_dataset = dpo_dataset.filter(
#     lambda r: (
#         r["status"] != "tie"
#         and r["chosen_score"] >= 8
#         and not r["in_gsm8k_train"]
#     )
# )

# Apply format and remove original columns
# YOUR CODE HERE
# dpo_dataset = dpo_dataset.map(format_prompt, remove_columns=dpo_dataset.column_names)

# Inspect
# print(f"DPO dataset size: {len(dpo_dataset)}")
# dpo_dataset[0]

## 3.2 Load Quantized SFT Model

DPO builds on top of the SFT model. We load the saved `TinyLlama-1.1B-qlora` adapter, merge it into the base model, and then re-apply quantization for efficient DPO training.

**Task:** Load `TinyLlama-1.1B-qlora` with `AutoPeftModelForCausalLM`, call `merge_and_unload()` to get the SFT-merged model, then apply the same 4-bit `BitsAndBytesConfig` as in section 1.2.

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig, AutoTokenizer

# YOUR CODE HERE — same bnb_config as section 1.2
# bnb_config = BitsAndBytesConfig(...)

# Load the SFT adapter and merge into the base model
# YOUR CODE HERE
# model = AutoPeftModelForCausalLM.from_pretrained(
#     "TinyLlama-1.1B-qlora",
#     low_cpu_mem_usage=True,
#     device_map="auto",
#     quantization_config=bnb_config,
# )
# model = model.merge_and_unload()   # SFT weights now baked in

# Reload tokenizer
# YOUR CODE HERE
# model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
# tokenizer.pad_token = "<PAD>"
# tokenizer.padding_side = "left"

## 3.3 LoRA Configuration for DPO

The LoRA config for DPO is identical to SFT — same rank, same target modules. A fresh set of A and B matrices will be trained on top of the already-merged SFT weights.

**Task:** Re-create the same `LoraConfig` as in section 1.3 and prepare the model.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# YOUR CODE HERE — same as section 1.3
# peft_config = LoraConfig(...)
# model = prepare_model_for_kbit_training(model)
# model = get_peft_model(model, peft_config)

## 3.4 DPO Training Configuration

DPO uses `DPOConfig` instead of `TrainingArguments`. Two parameters are **different** from SFT — understand why before writing the code:

| Parameter | SFT value | DPO value | Reason |
|-----------|-----------|-----------|--------|
| `learning_rate` | 2e-4 | **1e-5** | DPO makes subtle adjustments; large updates would destroy SFT |
| `warmup_ratio` | not set | **0.1** | Stabilise gradients for the first 10% of steps |
| `max_steps` | not set | **200** | Short run for illustration; real training uses full epoch |

**Task:** Create the `DPOConfig` with these values plus the shared parameters (batch_size=2, grad_accumulation=4, paged_adamw_32bit, cosine scheduler, fp16, gradient_checkpointing).

In [ ]:
from trl import DPOConfig

# YOUR CODE HERE
# training_arguments = DPOConfig(
#     output_dir="./results",
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     optim="paged_adamw_32bit",
#     learning_rate=1e-5,          # 20× lower than SFT
#     lr_scheduler_type="cosine",
#     max_steps=200,
#     logging_steps=10,
#     fp16=True,
#     gradient_checkpointing=True,
#     warmup_ratio=0.1,            # warm up for first 10% of steps
# )

## 3.5 Train with DPOTrainer

The `DPOTrainer` takes a `beta` parameter — the key DPO hyperparameter. It controls how much the trainable model is allowed to diverge from the frozen reference model.

- **Small β (0.1):** Large divergence allowed — stronger preference alignment, but risk of forgetting SFT behaviour
- **Large β (0.5+):** Conservative — small adjustments, stays close to SFT output

**Task:** Create a `DPOTrainer` with `beta=0.1`, then train and save the adapter.

In [ ]:
from trl import DPOTrainer

# YOUR CODE HERE
# dpo_trainer = DPOTrainer(
#     model,
#     args=training_arguments,
#     train_dataset=dpo_dataset,
#     tokenizer=tokenizer,
#     peft_config=peft_config,
#     beta=0.1,
#     max_prompt_length=512,
#     max_length=512,
# )

# dpo_trainer.train()
# dpo_trainer.model.save_pretrained("TinyLlama-1.1B-dpo-qlora")

## 3.6 Stack and Merge Both Adapters

The full pipeline stacks two LoRA adapters:
1. Load the SFT adapter → merge into base → get SFT-merged model
2. Load the DPO adapter on top of the SFT-merged model → merge again

The result is a single standard model with both SFT and DPO baked in — no adapter overhead at inference.

**Task:** Implement the two-step merge pipeline.

In [ ]:
from peft import AutoPeftModelForCausalLM, PeftModel

# Step 1: Load SFT adapter and merge into base model
# YOUR CODE HERE
# model = AutoPeftModelForCausalLM.from_pretrained(
#     "TinyLlama-1.1B-qlora",
#     low_cpu_mem_usage=True,
#     device_map="auto",
# )
# sft_model = model.merge_and_unload()

# Step 2: Load DPO adapter on top of SFT-merged model and merge
# YOUR CODE HERE
# dpo_model = PeftModel.from_pretrained(
#     sft_model,
#     "TinyLlama-1.1B-dpo-qlora",
#     device_map="auto",
# )
# final_model = dpo_model.merge_and_unload()

## 3.7 Final Aligned Model Inference

**Task:** Run inference on the final model. Then run the **same prompts** on just the SFT model (from Part 1) and compare.

Look for:
- Does the DPO model give more detailed answers?
- Does it refuse or qualify unsafe questions more carefully?
- Does it match the preferred response style from the training data?

Also try the same prompt with different `temperature` values (0.1 = deterministic, 0.9 = creative) to understand how generation sampling interacts with preference tuning.

In [ ]:
from transformers import pipeline

prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# Run the final DPO-aligned model
# YOUR CODE HERE
# pipe = pipeline(task="text-generation", model=final_model, tokenizer=tokenizer)
# print("--- Final DPO model ---")
# print(pipe(prompt, max_new_tokens=200)[0]["generated_text"])

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

---
## Chapter 12 Summary

You have now implemented the complete two-stage fine-tuning pipeline for generation models:

| Stage | What it teaches the model | Training data | Key technique |
|-------|--------------------------|---------------|---------------|
| **SFT** | Follow instructions (not just complete text) | Instruction–response pairs (UltraChat) | QLoRA (4-bit base + float16 adapters) |
| **DPO** | Prefer better responses over worse ones | Chosen vs rejected pairs | Log-probability ratio loss, no reward model |

**Key numbers to remember:**
- LoRA at rank 64, d=2,048 (TinyLlama): ~**131K trainable params** vs 4B in full fine-tuning
- QLoRA memory: ~1 GB for 1.1B model vs ~4 GB without quantization
- DPO learning rate is 20× lower than SFT: preference alignment is subtle, not a wholesale transformation

**Key insight to carry forward:** The loss function and fine-tuning strategy are higher-leverage choices than raw data volume. QLoRA + DPO lets a single A6000 GPU do in hours what previously required a data-centre cluster.